# Clustering: MSE no Rel 100 

In [28]:
import os

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd

from astroExplain.spectra import astronomy
from sdss.metadata import MetaData

meta = MetaData()

# Custom functions

## Data ingestion

# Config

## Directories

In [1]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
ch_4_dir = f"{thesis_dir}/chapters/04_figures"

# bin_id = "bin_03"
# phd_dir = "/home/elom/phd"
# spectra_dir = f"{phd_dir}/00_phd_code/spectra"
# explanations_dir = f"{phd_dir}/00_phd_code/explanations"
# paper_figures_dir = f"{phd_dir}/00_paper_explain-me-why/sections/figures/"
# clusterin_dir = f"{phd_dir}/00_phd_code/clustering"
# wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
# # convert from angstrom to nanometer
# wave /= 10.0

## Data

In [6]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)

## SE + RSE scores

In [11]:
bin_id = "bin_03"
res_scores_03_df = pd.read_csv(
    f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
    index_col='specobjid'
)

rank = np.arange(res_scores_03_df.shape[0]) + 1
res_scores_rank_df = res_scores_03_df.copy()
# score_rank_df
for col in res_scores_03_df.columns:

    index_sorted = res_scores_03_df.sort_values(
        by=col, ascending=False
    ).index

    res_scores_rank_df.loc[index_sorted, f'rank_{col}'] = rank
    res_scores_rank_df[f'rank_{col}'].astype(int)

n_spec = res_scores_03_df.shape[0]
n_top_1_pct = int(n_spec*0.01)
n_top_1_pct, n_spec

(1818, 181850)

In [12]:
res_scores_rank_df

,mse_rel,mse_filter_250_97_rel,mse_95,mse_97_rel,mse,mse_97,mse_filter_250_95_rel,mse_filter_300_95,mse_95_rel,mse_filter_300,...,rank_mse_95_rel,rank_mse_filter_300,rank_mse_filter_250_97,rank_mse_filter_300_95_rel,rank_mse_filter_250,rank_mse_filter_300_97,rank_mse_filter_250_95,rank_mse_filter_300_97_rel,rank_mse_filter_300_rel,rank_mse_filter_250_rel
specobjid,,,,,,,,,,,,,,,,,,,,,
1437819582281705472,3.549991,2.582845,2.446672,2.660625,3.570211,2.631072,2.392643,2.349896,2.443552,3.313081,...,20930.0,7140.0,10084.0,22490.0,7623.0,10152.0,10706.0,21231.0,15371.0,15236.0
1792473383787587584,3.592197,2.827291,2.530246,2.874075,3.230481,2.715585,2.589435,2.441902,2.649674,3.171808,...,7072.0,10176.0,6116.0,6051.0,11194.0,6310.0,5624.0,7568.0,9937.0,11277.0
1566270578824865792,3.799695,2.967733,2.379282,3.016445,3.215022,2.564632,2.679994,2.374952,2.759701,3.109789,...,4101.0,11978.0,11560.0,5437.0,11972.0,11425.0,11801.0,3233.0,6441.0,6498.0
1833056640966879232,3.418285,2.751345,2.359191,2.802071,2.965650,2.507619,2.559024,2.304638,2.579787,2.913956,...,10205.0,20428.0,14928.0,7485.0,20074.0,14313.0,13886.0,8767.0,16590.0,16849.0
2824940385879484416,3.393606,2.592578,2.230282,2.649388,2.839407,2.384083,2.394114,2.185068,2.418753,2.778646,...,23675.0,29717.0,24506.0,20963.0,29881.0,24741.0,22799.0,21441.0,16841.0,16922.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1874695412833609728,1.912835,1.531036,1.289596,1.475990,1.893698,1.386869,1.418995,1.252631,1.362300,1.594384,...,164001.0,158471.0,159602.0,161457.0,157069.0,163690.0,153091.0,160564.0,166646.0,163442.0
1874775677182437376,1.767668,0.969841,0.822902,1.112145,1.478349,1.037690,1.046248,0.910222,1.079419,1.273083,...,177839.0,175037.0,174983.0,175057.0,176995.0,177581.0,179870.0,176509.0,175537.0,167006.0
1874654181147568128,1.857279,1.124122,0.890098,1.140682,1.540927,1.101522,1.107023,0.860658,1.100273,1.281948,...,177195.0,174725.0,176521.0,173688.0,169745.0,180510.0,178525.0,179650.0,164724.0,170887.0


## Lines metadata from SDSS pipeline

In [14]:
lines_df = pd.read_csv(
    f"{spectra_dir}/meta_data/lines_EdgarOrtiz.csv",
    index_col="specObjID",
    low_memory=False
)
lines_df.shape

(1477411, 38)

## inner join anomalies and lines

In [15]:
cols_metadata = [
    'mjd', 'plate', 'fiberid', 'run2d', 'ra', 'dec',
    'z', 'zErr', 'zWarning', 'z_noqso', 'zErr_noqso','zWarning_noqso',
    'class', 'subClass',  'targetType', 'programname', 'instrument',
    'snMedian', 'ABSSB', 'BROAD', 'ebv',
    'score', 'cluster'
]
res_lines_df = res_scores_rank_df.join(lines_df, how='inner')
print(res_lines_df.shape)
res_lines_df.columns

(181457, 74)


Index(['mse_rel', 'mse_filter_250_97_rel', 'mse_95', 'mse_97_rel', 'mse',
       'mse_97', 'mse_filter_250_95_rel', 'mse_filter_300_95', 'mse_95_rel',
       'mse_filter_300', 'mse_filter_250_97', 'mse_filter_300_95_rel',
       'mse_filter_250', 'mse_filter_300_97', 'mse_filter_250_95',
       'mse_filter_300_97_rel', 'mse_filter_300_rel', 'mse_filter_250_rel',
       'rank_mse_rel', 'rank_mse_filter_250_97_rel', 'rank_mse_95',
       'rank_mse_97_rel', 'rank_mse', 'rank_mse_97',
       'rank_mse_filter_250_95_rel', 'rank_mse_filter_300_95',
       'rank_mse_95_rel', 'rank_mse_filter_300', 'rank_mse_filter_250_97',
       'rank_mse_filter_300_95_rel', 'rank_mse_filter_250',
       'rank_mse_filter_300_97', 'rank_mse_filter_250_95',
       'rank_mse_filter_300_97_rel', 'rank_mse_filter_300_rel',
       'rank_mse_filter_250_rel', 'oii_3726_flux', 'oii_3726_flux_err',
       'oii_3729_flux', 'oii_3729_flux_err', 'neiii_3869_flux',
       'neiii_3869_flux_err', 'h_delta_flux', 'h_delta_fl

In [16]:
res_lines_df.head()

,mse_rel,mse_filter_250_97_rel,mse_95,mse_97_rel,mse,mse_97,mse_filter_250_95_rel,mse_filter_300_95,mse_95_rel,mse_filter_300,...,sii_6717_flux,sii_6717_flux_err,sii_6731_flux,sii_6731_flux_err,ariii7135_flux,ariii7135_flux_err,oii_flux,oii_flux_err,oiii_flux,oiii_flux_err
1437819582281705472,3.549991,2.582845,2.446672,2.660625,3.570211,2.631072,2.392643,2.349896,2.443552,3.313081,...,121.926300,2.996880,90.058200,2.797693,7.585026,2.344169,256.730200,11.777070,76.773120,3.482531
1792473383787587584,3.592197,2.827291,2.530246,2.874075,3.230481,2.715585,2.589435,2.441902,2.649674,3.171808,...,4.732165,1.222621,4.872820,1.298028,0.907843,1.253644,13.600400,4.315449,9.062179,3.244147
1566270578824865792,3.799695,2.967733,2.379282,3.016445,3.215022,2.564632,2.679994,2.374952,2.759701,3.109789,...,12.443120,4.653118,-3.415565,4.730724,-4.969798,3.635871,-4.060645,5.168249,6.602443,3.662050
1833056640966879232,3.418285,2.751345,2.359191,2.802071,2.965650,2.507619,2.559024,2.304638,2.579787,2.913956,...,-0.863091,3.255430,-0.658119,2.008540,2.144301,3.464672,-4.017598,8.294394,-1.083071,1.528086
2824940385879484416,3.393606,2.592578,2.230282,2.649388,2.839407,2.384083,2.394114,2.185068,2.418753,2.778646,...,-1.261499,3.161689,-2.924345,3.203567,-6.914393,3.327213,10.926950,14.533940,12.422610,3.798780


# EDA fluxes

In [17]:
lines_cols = [
    "h_alpha_flux", "h_beta_flux",
    "nii_6548_flux", "nii_6584_flux",
    "sii_6717_flux", "sii_6731_flux",
    "oii_3726_flux", "oii_3729_flux",
    "oiii_4959_flux", "oiii_5007_flux", "oiii_flux" 
]
res_lines_df[lines_cols].describe()

,h_alpha_flux,h_beta_flux,nii_6548_flux,nii_6584_flux,sii_6717_flux,sii_6731_flux,oii_3726_flux,oii_3729_flux,oiii_4959_flux,oiii_5007_flux,oiii_flux
count,1.814570e+05,1.814570e+05,1.814570e+05,1.814570e+05,1.814570e+05,1.814570e+05,1.814570e+05,1.814570e+05,1.814570e+05,1.814570e+05,1.814570e+05
mean,8.975622e+05,2.319494e+05,4.411457e+05,1.330598e+06,-1.246196e+06,2.415075e+06,7.000327e+06,2.079458e+07,6.330987e+04,1.538721e+05,1.567924e+05
std,1.051919e+08,2.735681e+07,1.213879e+08,3.661340e+08,6.185059e+08,8.294610e+08,1.463272e+09,3.333411e+09,9.658346e+06,2.334485e+07,2.345455e+07
min,-9.832831e+08,-5.777803e+06,-2.264222e+07,-6.829417e+07,-2.631785e+11,-7.180114e+10,-1.796925e+11,-2.694684e+11,-1.476749e+08,-6.166620e+06,-6.328831e+06
25%,1.385621e+01,4.933405e+00,3.133515e+00,9.451404e+00,2.217676e+00,6.988558e-01,2.301679e+00,1.622084e+00,5.715988e-01,7.543362e+00,7.938534e+00
50%,3.780022e+01,1.289014e+01,1.087748e+01,3.280900e+01,1.284905e+01,8.500538e+00,1.487862e+01,1.481151e+01,4.736154e+00,1.562750e+01,1.621325e+01
75%,1.791501e+02,4.342221e+01,3.411325e+01,1.028934e+02,4.271759e+01,3.088515e+01,3.901466e+01,3.822337e+01,1.096004e+01,3.171705e+01,3.266922e+01
max,2.292654e+10,6.261624e+09,5.080869e+10,1.532508e+11,9.422665e+09,2.973184e+11,3.091597e+11,1.087276e+12,2.556557e+09,7.612909e+09,7.590690e+09


In [18]:
# print("Number of NaNs in each line flux column:")
n_nans = res_lines_df[lines_cols].isna().sum()
# print(n_nans)
# print("Number of negative flux values in each line flux column:")
n_negatives = (res_lines_df[lines_cols] < 0).sum()
# print(n_negatives)
# print("Number of zero flux values in each line flux column:")
n_zeros = (res_lines_df[lines_cols] == 0).sum()
# print(n_zeros)
# add n_zeros, n_nans, n_negatives to a temporary DataFrame
temp_df = pd.DataFrame({
    "n_nans": n_nans,
    "n_negatives": n_negatives,
    "n_zeros": n_zeros
})
temp_df.index.name = "line_flux_column"
temp_df = temp_df.reset_index()
temp_df

,line_flux_column,n_nans,n_negatives,n_zeros
0,h_alpha_flux,0,3293,502
1,h_beta_flux,0,12432,280
2,nii_6548_flux,0,10664,420
3,nii_6584_flux,0,10664,420
4,sii_6717_flux,0,30392,535
5,sii_6731_flux,0,39377,531
6,oii_3726_flux,0,29249,5864
7,oii_3729_flux,0,33043,5332
8,oiii_4959_flux,0,39494,278
9,oiii_5007_flux,0,7368,280


In [22]:
# # Compute deciles (10th to 90th percentiles) for each column
# deciles = np.arange(0, 1.01, 0.1)
# decile_df = res_lines_df[lines_cols].quantile(q=deciles).T
# decile_df.columns = [f"{int(q * 100)}th" for q in deciles]
# decile_df

## Data prep to compute ratios

In [23]:
# replace 0 with NaN
res_lines_df[lines_cols] = res_lines_df[lines_cols].replace(0, np.nan)
# replace negative fluxes numeric values with NaN
res_lines_df[lines_cols] = res_lines_df[lines_cols].where(
    res_lines_df[lines_cols] >= 0,
    np.nan
)
n_zeros = (res_lines_df[lines_cols] == 0).sum()
n_negatives = (res_lines_df[lines_cols] < 0).sum()
temp_df = pd.DataFrame({
    "n_negatives": n_negatives,
    "n_zeros": n_zeros
})
temp_df.index.name = "line_flux_column"
temp_df = temp_df.reset_index()
temp_df

,line_flux_column,n_negatives,n_zeros
0,h_alpha_flux,0,0
1,h_beta_flux,0,0
2,nii_6548_flux,0,0
3,nii_6584_flux,0,0
4,sii_6717_flux,0,0
5,sii_6731_flux,0,0
6,oii_3726_flux,0,0
7,oii_3729_flux,0,0
8,oiii_4959_flux,0,0
9,oiii_5007_flux,0,0


## Median line ratios

In [29]:
df = astronomy.compute_emission_line_ratios(res_lines_df)

In [30]:
df.shape

(181457, 81)

In [31]:
ratios_cols = [
    "balmer_decrement",
    "nii_to_halpha",
    "oiii_to_hbeta",
    "oiii_to_oii",
    "o3n2_index",
    # "sii_to_halpha",
    # "sii_density_ratio",
]
# Compute percentiles (45th to 55th percentiles) for each column
percentiles = np.arange(0.45, 0.55, 0.01)
# Create a DataFrame of percentiles for each column in fluxes_df[lines_cols]
ratios_percentiles_df = df[ratios_cols].quantile(q=percentiles).T
# Optionally rename the rows as "10th", "20th", etc.
ratios_percentiles_df.columns = [f"{int(q * 100)}th" for q in percentiles]
# Display the result
ratios_percentiles_df

,45th,46th,47th,48th,49th,50th,51th,52th,53th,54th,55th
balmer_decrement,3.548007,3.588622,3.627285,3.668651,3.707243,3.744923,3.782970,3.820659,3.858997,3.895802,3.932153
nii_to_halpha,0.583414,0.595367,0.607766,0.619722,0.631918,0.645449,0.658609,0.673131,0.687132,0.701055,0.715934
oiii_to_hbeta,0.962152,0.990143,1.017523,1.043962,1.071854,1.100840,1.129460,1.158159,1.187582,1.218355,1.249103
oiii_to_oii,0.766708,0.780790,0.794740,0.808733,0.823250,0.838865,0.853888,0.869229,0.885436,0.902445,0.919984
o3n2_index,0.117505,0.126425,0.135576,0.144630,0.153545,0.162499,0.172088,0.181370,0.190887,0.200544,0.210003


# Common anomalies

In [36]:
overview_common_dict = {
    # row 1
    'narrow_line': 734111514142730240,
    'broad_line_large_OIII':1633733043925575680, 
    # row 2
    'broad_emission_dips_OIII_half': 1192355533059811328,
    'star_forming_step_blue_slope': 1959124163192973312,
    # row 3
    'blue_bump_emission': 531492683672217600,
    'passive_star': 1780176998165932032,
    # row 4
    'spike':1413149843194406912,
    'noise_forest': 637325355518027776,

}

In [37]:
ratios_cols = [
    "balmer_decrement",
    "nii_to_halpha",
    "oiii_to_hbeta",
    "oiii_to_oii",
    "o3n2_index",
    "sii_to_halpha",
    "sii_density_ratio",
]
df.loc[734111514142730240, ratios_cols]

balmer_decrement     3.293360
nii_to_halpha        0.050160
oiii_to_hbeta        3.990424
oiii_to_oii               NaN
o3n2_index           1.900664
sii_to_halpha        0.163502
sii_density_ratio    1.362553
Name: 734111514142730240, dtype: float64

In [43]:
ratios_dict = {
    "specobjid": [],
    "name": [],
    "balmer_decrement": [],
    "nii_to_halpha": [],
    "oiii_to_hbeta": [],
    "oiii_to_oii": [],
    "o3n2_index": [],
    "sii_to_halpha": [],
    "sii_density_ratio": []
}


for title, specid in overview_common_dict.items():
    ratios_dict["name"].append(title)
    ratios_dict["specobjid"].append(specid)
    for col in ratios_cols:
        try:
            ratios_dict[col].append(df.loc[specid, col])
        except KeyError:
            ratios_dict[col].append(np.nan)

In [47]:
common_anomalies_ratios_df = pd.DataFrame(ratios_dict)
common_anomalies_ratios_df.to_clipboard()
common_anomalies_ratios_df

,specobjid,name,balmer_decrement,nii_to_halpha,oiii_to_hbeta,oiii_to_oii,o3n2_index,sii_to_halpha,sii_density_ratio
0,734111514142730240,narrow_line,3.293360,0.050160,3.990424,NaN,1.900664,0.163502,1.362553
1,1633733043925575680,broad_line_large_OIII,3.891876,0.204359,10.550430,11.752099,1.712876,0.217171,1.109206
2,1192355533059811328,broad_emission_dips_OIII_half,5.013954,0.599002,8.754367,7.460670,1.164797,0.344767,1.184883
3,1959124163192973312,star_forming_step_blue_slope,3.437888,0.182478,1.409797,0.595381,0.887945,0.278217,1.376201
4,531492683672217600,blue_bump_emission,3.584317,0.371369,0.517071,0.429446,0.143744,0.333807,1.389607
5,1780176998165932032,passive_star,NaN,NaN,NaN,0.123804,NaN,NaN,NaN
6,1413149843194406912,spike,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,637325355518027776,noise_forest,5.763929,0.055846,0.861052,0.411711,1.188040,NaN,NaN


In [50]:
class_common_anomalies_dict = {
    "specobjid": [],
    "name": [],
    "class": [], 
}

for title, specid in overview_common_dict.items():

    class_common_anomalies_dict["name"].append(title)
    class_common_anomalies_dict["specobjid"].append(specid)
    try:
        class_ = final_meta_df.loc[specid, "subClass"]
        class_common_anomalies_dict["class"].append(class_)
    except KeyError:
        class_common_anomalies_dict["class"].append(np.nan)

class_common_anomalies_df = pd.DataFrame(class_common_anomalies_dict)
class_common_anomalies_df.to_clipboard()
class_common_anomalies_df


,specobjid,name,class
0,734111514142730240,narrow_line,STARBURST
1,1633733043925575680,broad_line_large_OIII,STARBURST
2,1192355533059811328,broad_emission_dips_OIII_half,AGN BROADLINE
3,1959124163192973312,star_forming_step_blue_slope,STARBURST
4,531492683672217600,blue_bump_emission,STARFORMING
5,1780176998165932032,passive_star,NaN
6,1413149843194406912,spike,NaN
7,637325355518027776,noise_forest,NaN
